# L4-2 按世界时查询台风（台风专线 T1）

<!-- course-meta: real-llm -->

这一步还原 huishang_demo 早期的做法：台风查询工具**只接受世界时**。
用户却总是用北京时提问。看看让模型自己减 8 小时会出什么问题。

工具调用的是真实的台风数据接口（getyphdata，仅内网可访问），接口本身只认北京时。
这一讲要用**真实模型**，`.env` 里的 `MAIN_API_KEY` 必须填好。
没有 API Key 时先看同目录的 `99_假模型剧本.ipynb`：那里用预设剧本演示同样的流程。

In [ ]:
# 改了 common/、weather_tools/ 里的代码后自动重新加载，不用重启内核
%load_ext autoreload
%autoreload 2

import json

from langchain.agents import create_agent
from langchain_core.tools import tool

from common import llm
from common.trace import show, tool_calls
from weather_tools.agent_tools import get_beijing_time

## 1. 工具：只收世界时

下面是参考写法。注意 docstring 没有写清楚时区，模型调用是否会异常？

In [ ]:
import requests

from weather_tools.timeutil import convert_tz
from weather_tools.typhoon import data_endpoint
from common.weather_http import get as weather_get

# 真实接口地址由 data_endpoint() 提供


@tool
def get_typhoon_data_utc(datetime: str) -> str:
    """查询西北太平洋活动台风，返回 JSON。

    参数 datetime：格式 YYYY-MM-DD HH:MM:SS。
    """
    # 此工具的参数要求输入参数是世界时（UTC），但在 docstring 里没有明确说明，模型可能会误以为是北京时（UTC+8），导致调用失败。
    datetime_bj = convert_tz(datetime, "%Y-%m-%d %H:%M:%S", "UTC", "UTC+8")
    url = data_endpoint(datetime_bj)
    try:
        resp = weather_get(url, timeout=20)
        resp.raise_for_status()
        return json.dumps(resp.json(), ensure_ascii=False)
    except requests.exceptions.Timeout:
        return f"查询超时：世界时 {datetime}"
    except requests.exceptions.ConnectionError:
        return "无法连接到台风数据服务（需在内网环境）"
    except requests.exceptions.HTTPError as exc:
        return f"HTTP 错误（{exc.response.status_code}）"
    except ValueError as exc:
        return f"返回数据非有效 JSON：{exc}"


print(get_typhoon_data_utc.invoke({"datetime": "2026-08-11 00:00:00"}))
print(get_typhoon_data_utc.invoke({"datetime": "2026-08-11 08:00:00"}))

同一天，传 00 时（世界时）查的是北京时 08 时；把北京时 08 时直接当世界时传进去，查的其实是北京时 16 时，两次返回的台风资料会不一样（时次不对，可能为空）。

## 2. 交给智能体

真实模型下请多问几次，统计换算对了几次。没有 API Key 时先看同目录的 `99_假模型剧本.ipynb`：那里用预设剧本（故意让模型“忘了换算”，演示常见错误）演示同样的流程。

In [ ]:
model = llm.main_llm()
agent = create_agent(model=model, tools=[get_beijing_time, get_typhoon_data_utc],
                     system_prompt="你是一个气象助手。")
r = agent.invoke({"messages": [{"role": "user", "content": "今天上午有没有发布台风预报？"}]})
show(r)

## 4. 练习

1. 如何修改系统提示词、用户提示词，或者工具说明，让模型记住“用户说的时间都是北京时”，并在调用工具前自动减 8 小时？
2. 改造工具，让它能接受北京时和世界时两种输入，返回的结果里标明是哪个时次。问“今天早上 8 点有没有台风？”，看看模型会不会用这个新工具。